In [1]:
cd ..

/Users/martinsevilla/Downloads/congestion-bogota


In [2]:
import os
import pandas as pd
from torch.utils.data import DataLoader

from src.dataloader import GraphTimeSeriesLoader, GraphTimeSeriesDataset

%load_ext autoreload
%autoreload 2

In [3]:
processed_data_dir = "data/datos_procesados"
dataset_filenames = {
    "buildings": f"{processed_data_dir}/buildings/roads_buildings.csv",
    "bus_stops": f"{processed_data_dir}/bus_stops/bus_stops_per_road.parquet",
    "events": f"{processed_data_dir}/events/events_dataset.csv",
    "incidents": f"{processed_data_dir}/incidents/incidents_dataset.csv",
    "irregularities": [f"{processed_data_dir}/irregularidades_procesadas/" + f for f in os.listdir(f"{processed_data_dir}/irregularidades_procesadas/") if f.endswith('.parquet')],
    # "coi_points": "pmts_processed/coi_puntos_dataset.parquet",
    # "coi_parks": "pmts_processed/coi_parques_dataset.parquet",
    "coi_points": [f"{processed_data_dir}/pmts_processed/coi_puntos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_puntos/") if f.endswith('.parquet')],
    "coi_tracts": [f"{processed_data_dir}/pmts_processed/coi_tramos/" + f for f in os.listdir(f"{processed_data_dir}/pmts_processed/coi_tramos/") if f.endswith('.parquet')],
    "road_state": f"{processed_data_dir}/road_state/road_state_dateset.parquet", # TODO: Should be 'dataset' no 'dateset'
}
graph_filename = f"{processed_data_dir}/conectivity_graph/road_graph.pickle"
imputation_parquet_filename = f"{processed_data_dir}/neighbor_features.parquet"

# Load one file from each dataset to see the column names and their types
def load_dataset_columns(dataset_filenames):
    columns: dict[str, str] = {}
    for key, filename in dataset_filenames.items():
        if isinstance(filename, list):
            # If it's a list, load the first file
            filename = filename[0]
        df = pd.read_parquet(filename) if filename.endswith('.parquet') else pd.read_csv(filename)
        columns.update({f"{key}_{col}": str(dtype) for col, dtype in zip(df.columns, df.dtypes)})
    return columns

dataset_columns = load_dataset_columns(dataset_filenames)
dataset_columns_df = pd.DataFrame(dataset_columns, index=["dtype"]).T.reset_index().rename(columns={"index": "column_name"})
object_columns = dataset_columns_df[~ dataset_columns_df["dtype"].isin(["float64", "int64", "int32"])].column_name

assert all("timeframe" in col or "road_id" in col or "id_mvi" in col for col in object_columns)

In [4]:
loader = GraphTimeSeriesLoader(
    graph_pkl_filename=graph_filename,
    events_csv_filename=dataset_filenames["events"],
    incidents_csv_filename=dataset_filenames["incidents"],
    road_state_parquet_filename=dataset_filenames["road_state"],
    irregularities_filenames=dataset_filenames["irregularities"],
    coi_points_filenames=dataset_filenames["coi_points"],
    coi_tracts_filenames=dataset_filenames["coi_tracts"],
    default_speeds_parquet_filename=imputation_parquet_filename,
)

In [5]:
# incident_set = set([tuple(l) for l in loader.incidents_df[["road_id", "timeframe"]].values])
# event_set = set([tuple(l) for l in loader.events_df[["road_id", "timeframe"]].values])
# common_tuples = incident_set.intersection(event_set)
# list(common_tuples)[:5]

In [6]:
# Example usage of the loader to load data for specific nodes and time windows
nodes_to_load = ["9004322_TF.1", "50008506_TF.1", "12003262_FT.1"]
time_windows_to_load = [
    ('2024-09-29 23:00:00', 4, 5),
    ('2024-08-24 15:30:00', 4, 5),
    ('2024-02-24 01:30:00', 4, 5)
]

# Load the data
(static_features, temporal_features), target = loader.load_data(
    nodes_to_load, time_windows_to_load
)


--- Starting data load for 3 nodes ---


KeyboardInterrupt: 

In [ ]:
static_features

,mvi_numc,buildings_farmacia,buildings_posgrados_universidad_santo_tomas,buildings_abandoned,buildings_administration,buildings_apartments,buildings_apartments_commercial_garage,buildings_bakehouse,buildings_barn,buildings_bridge,...,mvi_tipo_ac,mvi_tipo_ak,mvi_tipo_cl,mvi_tipo_dg,mvi_tipo_kr,mvi_tipo_tv,mvi_tipo_nan,mvi_velreg_30,mvi_velreg_50,mvi_velreg_nan
road_id,,,,,,,,,,,,,,,,,,,,,
9004322_TF.1,2.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,1
50008506_TF.1,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1,0,0,0,0,0,0,0,0,1
12003262_FT.1,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,1,0,0,0,0,0,0,1,0


In [ ]:
temporal_features.head(5)

aglomeracion  emergencia  manifestacion  \
road_id      timeframe                                                      
9004322_TF.1 2024-02-24 00:15:00           0.0         0.0            0.0   
             2024-02-24 00:30:00           0.0         0.0            0.0   
             2024-02-24 00:45:00           0.0         0.0            0.0   
             2024-02-24 01:00:00           0.0         0.0            0.0   
             2024-02-24 01:15:00           0.0         0.0            0.0   

                                  incident_type_703  incident_type_715  \
road_id      timeframe                                                   
9004322_TF.1 2024-02-24 00:15:00                0.0                0.0   
             2024-02-24 00:30:00                0.0                0.0   
             2024-02-24 00:45:00                0.0                0.0   
             2024-02-24 01:00:00                0.0                0.0   
             2024-02-24 01:15:00                0.0                0.0   

                                  incident_type_802  incident_type_901  \
road_id      timeframe                                                   
9004322_TF.1 2024-02-24 00:15:00                0.0                0.0   
             2024-02-24 00:30:00                0.0                0.0   
             2024-02-24 00:45:00                0.0                0.0   
             2024-02-24 01:00:00                0.0                0.0   
             2024-02-24 01:15:00                0.0                0.0   

                                  incident_type_910  incident_type_914  \
road_id      timeframe                                                   
9004322_TF.1 2024-02-24 00:15:00                0.0                0.0   
             2024-02-24 00:30:00                0.0                0.0   
             2024-02-24 00:45:00                0.0                0.0   
             2024-02-24 01:00:00                0.0                0.0   
             2024-02-24 01:15:00                0.0                0.0   

                                  incident_type_929  ...  n_thumbs_up  \
road_id      timeframe                               ...                
9004322_TF.1 2024-02-24 00:15:00                0.0  ...            0   
             2024-02-24 00:30:00                0.0  ...            0   
             2024-02-24 00:45:00                0.0  ...            0   
             2024-02-24 01:00:00                0.0  ...            0   
             2024-02-24 01:15:00                0.0  ...            0   

                                  seconds  cause_type  cause_type_0  \
road_id      timeframe                                                
9004322_TF.1 2024-02-24 00:15:00        0           0             0   
             2024-02-24 00:30:00        0           0             0   
             2024-02-24 00:45:00        0           0             0   
             2024-02-24 01:00:00        0           0             0   
             2024-02-24 01:15:00        0           0             0   

                                  cause_type_accident  cause_type_hazard  \
road_id      timeframe                                                     
9004322_TF.1 2024-02-24 00:15:00                    0                  0   
             2024-02-24 00:30:00                    0                  0   
             2024-02-24 00:45:00                    0                  0   
             2024-02-24 01:00:00                    0                  0   
             2024-02-24 01:15:00                    0                  0   

                                  cause_type_road_closed  severity  \
road_id      timeframe                                               
9004322_TF.1 2024-02-24 00:15:00                       0         0   
             2024-02-24 00:30:00                       0         0   
             2024-02-24 00:45:00                       0         0   
             2024-02-24 01:00:00            

In [ ]:
target.head(5)

road_id       timeframe          
9004322_TF.1  2024-02-24 00:15:00    0
              2024-02-24 00:30:00    0
              2024-02-24 00:45:00    0
              2024-02-24 01:00:00    0
              2024-02-24 01:15:00    0
Name: jam_level, dtype: int64

In [7]:
# Create an instance of the custom dataset
train_dataset = GraphTimeSeriesDataset(loader=loader, node_ids=nodes_to_load, time_windows=time_windows_to_load)

# Create the DataLoader
# This will handle batching, shuffling, and parallel data loading.
batch_size = 2
train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,  # Shuffle the data at the beginning of each epoch
    num_workers=0   # Set to > 0 to use multiprocessing for data loading
)

# Iterate over the DataLoader in the training loop
num_epochs = 3
for epoch in range(num_epochs):
    print(f"--- Epoch {epoch+1}/{num_epochs} ---")
    for i, (static_batch, temporal_batch, target_batch) in enumerate(train_dataloader):
        print(f"  Batch {i+1}:")
        print(f"    Static data shape: {static_batch.shape}")
        print(f"    Temporal data shape: {temporal_batch.shape}")
        print(f"    Target data shape: {target_batch.shape}")
        print("-" * 20)

        # Here the batches should be fed to the GNN model for training
        # model.train()
        # optimizer.zero_grad()
        # output = model(static_batch, temporal_batch)
        # loss = criterion(output, target_batch)
        # loss.backward()
        # optimizer.step()

--- Epoch 1/3 ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (30, 112) ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (30, 112) ---
  Batch 1:
    Static data shape: torch.Size([2, 3, 112])
    Temporal data shape: torch.Size([2, 30, 109])
    Target data shape: torch.Size([2, 30])
--------------------

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (30, 112) ---
  Batch 2:
    Static data shape: torch.Size([1, 3, 112])
    Temporal data shape: torch.Size([1, 30, 109])
    Target data shape: torch.Size([1, 30])
--------------------
--- Epoch 2/3 ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (30, 112) ---

--- Starting data load for 3 nodes ---
--- Load complete. Found 3 static records. Temporal DF shape: (30, 112) ---
  Batch 1:
    Static data shape: torch.